# ARTI 308 – Lab 5: Feature Engineering (Classification)

## Student Health Dataset Version

## 1. Setup and imports

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.ensemble import RandomForestClassifier

sns.set(style="whitegrid")
pd.set_option("display.max_columns", None)

## 2. Load the dataset

In [ ]:
DATA_PATH = "student_health_data.csv"  # ensure the file is in the same folder as this notebook

def load_student_health_dataset(path):
    # The provided file has a semicolon header, but each row stores the real values as one comma-separated block.
    # This loader reconstructs the dataset into a clean table.
    columns = [
        "Student_ID", "Age", "Gender", "Heart_Rate", "Blood_Pressure_Systolic",
        "Blood_Pressure_Diastolic", "Stress_Level_Biosensor", "Stress_Level_Self_Report",
        "Physical_Activity", "Sleep_Quality", "Mood", "Study_Hours", "Project_Hours",
        "Health_Risk_Level", "Family_members", "Has_Chronic_Condition", "Has_Allergies"
    ]

    rows = []
    with open(path, "r", encoding="utf-8") as f:
        _ = next(f)  # skip original header
        for line in f:
            main_part = line.split(";")[0].strip()
            values = main_part.split(",")
            values = values[:17]
            values += [""] * (17 - len(values))
            rows.append(values)

    df = pd.DataFrame(rows, columns=columns)

    numeric_cols = [
        "Student_ID", "Age", "Heart_Rate", "Blood_Pressure_Systolic",
        "Blood_Pressure_Diastolic", "Stress_Level_Biosensor",
        "Stress_Level_Self_Report", "Study_Hours", "Project_Hours", "Family_members"
    ]

    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    for col in ["Gender", "Physical_Activity", "Sleep_Quality", "Mood", "Health_Risk_Level"]:
        df[col] = df[col].replace("", "Unknown").fillna("Unknown")

    for col in ["Has_Chronic_Condition", "Has_Allergies"]:
        df[col] = df[col].replace("", "Unknown").fillna("Unknown")

    return df

df = load_student_health_dataset(DATA_PATH)
df.head(10)

The first rows confirm that the dataset loaded correctly after fixing the original file formatting.

## 3. Quick dataset checks (cleanliness confirmation)

In [ ]:
print("Shape:", df.shape)
print("\nMissing values per column:")
display(df.isna().sum().to_frame("missing_count").T)

print("\nDuplicate rows:", df.duplicated().sum())

We confirm the dataset is clean after reconstruction: no missing values and no duplicated rows.

## 4. Target variable and class balance

In [ ]:
target_col = "Health_Risk_Level"
df[target_col].value_counts()

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x=target_col, data=df, order=df[target_col].value_counts().index)
plt.title("Health Risk Level Distribution")
plt.xlabel("Health Risk Level")
plt.ylabel("Count")
plt.show()

This bar chart shows whether the classes are balanced. The dataset is somewhat imbalanced because the `Moderate` class appears more often than `Low` or `High`.

## 5. Identify feature types

In [ ]:
df.dtypes

We have a mixture of numerical features (for example, heart rate, blood pressure, study hours, and project hours) and categorical features (for example, gender, physical activity, sleep quality, and mood).

## 6. Leakage awareness (important)

When designing a prediction task, we must avoid using features that would not be available at prediction time. In this dataset, most variables are valid descriptive predictors. However, `Student_ID` is only an identifier, so it should not be treated as a meaningful predictive feature.

## 7. Feature engineering

### 7.1 Blood pressure features

In [ ]:
df_fe = df.copy()

df_fe["pulse_pressure"] = df_fe["Blood_Pressure_Systolic"] - df_fe["Blood_Pressure_Diastolic"]
df_fe["mean_arterial_pressure"] = (
    2 * df_fe["Blood_Pressure_Diastolic"] + df_fe["Blood_Pressure_Systolic"]
) / 3

df_fe[[
    "Blood_Pressure_Systolic", "Blood_Pressure_Diastolic",
    "pulse_pressure", "mean_arterial_pressure"
]].head(10)

We created medically meaningful features from blood pressure values. `pulse_pressure` captures the gap between systolic and diastolic pressure, while `mean_arterial_pressure` summarizes the average pressure load.

### 7.2 Workload and stress features

In [ ]:
df_fe["total_workload_hours"] = df_fe["Study_Hours"] + df_fe["Project_Hours"]
df_fe["stress_gap"] = df_fe["Stress_Level_Self_Report"] - df_fe["Stress_Level_Biosensor"]
df_fe["stress_product"] = df_fe["Stress_Level_Self_Report"] * df_fe["Stress_Level_Biosensor"]
df_fe["study_project_ratio"] = df_fe["Study_Hours"] / (df_fe["Project_Hours"] + 1e-3)

df_fe[[
    "Study_Hours", "Project_Hours", "total_workload_hours",
    "Stress_Level_Biosensor", "Stress_Level_Self_Report",
    "stress_gap", "stress_product", "study_project_ratio"
]].head(10)

These engineered features describe academic workload and the relationship between sensor-based stress and self-reported stress.

### 7.3 Discretization and grouped categories

In [ ]:
df_fe["age_group"] = pd.cut(
    df_fe["Age"],
    bins=[0, 20, 23, 26, np.inf],
    labels=["teen", "young_adult", "adult", "older"]
)

df_fe["family_size_group"] = pd.cut(
    df_fe["Family_members"],
    bins=[-1, 2, 5, 10, np.inf],
    labels=["small", "medium", "large", "very_large"]
)

df_fe["heart_rate_band"] = pd.cut(
    df_fe["Heart_Rate"],
    bins=[0, 60, 80, 100, np.inf],
    labels=["low", "normal", "elevated", "high"]
)

df_fe[["Age", "age_group", "Family_members", "family_size_group", "Heart_Rate", "heart_rate_band"]].head(10)

Discretization converts continuous values into categories. These grouped features may help the classifier capture patterns more clearly than raw numbers alone.

## 8. Prepare features for modeling

We now select our predictors.

In [ ]:
drop_cols = [
    "Student_ID"  # identifier only
]

drop_cols = [c for c in drop_cols if c in df_fe.columns]

X = df_fe.drop(columns=drop_cols + [target_col])
y = df_fe[target_col]

print("X shape:", X.shape)
print("y shape:", y.shape)
X.head()

We prepared a feature matrix `X` and a target vector `y`.

## 9. Split into train and test sets

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

We use stratified splitting to keep class proportions similar in train and test sets.

## 10. Encoding and baseline model (Random Forest)

### Why encoding?
Machine learning models need numerical input. Categorical columns must therefore be encoded before training.

In [ ]:
categorical_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_cols = X_train.select_dtypes(include=[np.number, "bool"]).columns.tolist()

print("Categorical columns:", categorical_cols)
print("Numeric columns:", numeric_cols)

preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", "passthrough", numeric_cols),
    ]
)

rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced_subsample"
)

model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("rf", rf)
])

model

## 11. Train the model and evaluate

In [ ]:
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=model.classes_)
plt.figure(figsize=(6,4))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=model.classes_, yticklabels=model.classes_
)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

Accuracy gives a general sense of performance, but the classification report is more informative because it shows precision, recall, and F1-score for each health risk class.

## 12. Feature importance (What mattered the most?)

Random Forest provides a built-in feature importance score.

In [ ]:
ohe = model.named_steps["preprocess"].named_transformers_["cat"]
cat_feature_names = ohe.get_feature_names_out(categorical_cols) if len(categorical_cols) > 0 else np.array([])
all_feature_names = np.concatenate([cat_feature_names, np.array(numeric_cols)])

importances = model.named_steps["rf"].feature_importances_

fi = (
    pd.DataFrame({"feature": all_feature_names, "importance": importances})
    .sort_values("importance", ascending=False)
)

fi.head(15)

In [ ]:
plt.figure(figsize=(8,5))
top_n = 15
sns.barplot(data=fi.head(top_n), x="importance", y="feature")
plt.title(f"Top {top_n} Feature Importances (Random Forest)")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()

This chart helps us understand which original and engineered features contributed most to predicting `Health_Risk_Level`.

## 13. Optional: Feature selection using SelectFromModel

We can select a subset of features using model-based selection.

In [ ]:
from sklearn.feature_selection import SelectFromModel

selector = SelectFromModel(
    estimator=RandomForestClassifier(
        n_estimators=300, random_state=42, n_jobs=-1, class_weight="balanced_subsample"
    ),
    threshold="median"
)

model_fs = Pipeline(steps=[
    ("preprocess", preprocess),
    ("select", selector),
    ("rf", RandomForestClassifier(
        n_estimators=300, random_state=42, n_jobs=-1, class_weight="balanced_subsample"
    ))
])

model_fs.fit(X_train, y_train)
y_pred_fs = model_fs.predict(X_test)

print("Accuracy (with feature selection):", round(accuracy_score(y_test, y_pred_fs), 4))
print("\nClassification Report (with feature selection):")
print(classification_report(y_test, y_pred_fs))

If performance stays similar, feature selection may help simplify the model with minimal accuracy loss.

## 14. Student tasks

### Task 1
Describe which engineered features are likely to be the most useful for predicting `Health_Risk_Level`, and explain why.

### Task 2
Compare the baseline Random Forest model with the feature-selection version. Did the performance improve, decrease, or stay similar?

### Task 3
Suggest one additional health-related feature that could be engineered from the current dataset and explain why it may help classification.

## Wrap-up

In this lab, we loaded a student health dataset, repaired its formatting, explored class balance, engineered meaningful health and workload features, trained a Random Forest classifier, and interpreted model importance scores. This follows the same workflow used in the original lab, but adapted to the student health problem.